Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun pipeline ETL produksi pertama yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan tiga sumber data sekaligus.

In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris.
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk.
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis).
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [2]:
# Membuat SparkSession.
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, round

spark = SparkSession.builder \
    .appName("Tugas6_Naufal_ETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 08:44:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/28 08:44:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


A. EXTRACT (bobot 15%)

Baca ketiga sumber data (tugas6_transaksi.csv, tugas6_produk.json, tugas6_ulasan.csv) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan printSchema() masing-masing.

In [3]:
# Jemabatan lokal.
path_trx = "file://" + os.path.abspath("tugas6_transaksi.csv")
path_prd = "file://" + os.path.abspath("tugas6_produk.json")
path_rev = "file://" + os.path.abspath("tugas6_ulasan.csv")

# Membaca ketiga sumber data.
df_trx = spark.read.csv(path_trx, header=True, inferSchema=True)
df_prd = spark.read.json(path_prd)
df_rev = spark.read.csv(path_rev, header=True, inferSchema=True)

# Menampilkan jumlah baris dan skema masing-masing.
print(f"Jumlah baris transaksi: {df_trx.count()}")
df_trx.printSchema()

print(f"Jumlah baris produk: {df_prd.count()}")
df_prd.printSchema()

print(f"Jumlah baris ulasan: {df_rev.count()}")
df_rev.printSchema()

Jumlah baris transaksi: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Jumlah baris produk: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Jumlah baris ulasan: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



B. TRANSFORM — Penggabungan (bobot 25%)

Gabungkan ketiga DataFrame menjadi satu (order_id sebagai kunci ke ulasan, product_id sebagai kunci ke produk). Gunakan salah satu join yang tepat dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan salah satu join yang tepat dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom total_pendapatan (unit_terjual x harga).

In [4]:
# Join transaksi ke ulasan dengan LEFT JOIN dan ke produk dengan INNER JOIN.
df_gabung = df_trx.join(df_rev, on="order_id", how="left") \
                  .join(df_prd, on="product_id", how="inner")

# Tambahkan kolom total_pendapatan (unit_terjual x harga).
df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
df_gabung.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

C. TRANSFORM — Penanganan Data Kosong & Pengayaan (bobot 20%)

    Transaksi tanpa ulasan akan memiliki rating bernilai kosong (null) setelah salah satu join yang tepat — isi nilai kosong tersebut dengan angka 0 menggunakan salah satu function, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
    Tambahkan kolom ada_ulasan bernilai True/False (tidak boleh diisi manual satu satu), **sebelum** langkah na.fill()` di atas).

In [5]:
# Tambahkan kolom ada_ulasan SEBELUM proses na.fill()[cite: 8]
df_enriched = df_gabung.withColumn("ada_ulasan", col("rating").isNotNull())

# Tangani rating kosong dengan angka 0.
df_enriched = df_enriched.na.fill({"rating": 0})

df_enriched.select("order_id", "nama_produk", "kategori", "rating", "ada_ulasan", "total_pendapatan").show(10)

+--------+-----------+------------+------+----------+----------------+
|order_id|nama_produk|    kategori|rating|ada_ulasan|total_pendapatan|
+--------+-----------+------------+------+----------+----------------+
|     TX0|  Produk-21|  Elektronik|     1|      true|           50000|
|     TX1|   Produk-8|     Fashion|     0|     false|         1500000|
|     TX2|  Produk-25|  Elektronik|     2|      true|          100000|
|     TX3|   Produk-3|     Fashion|     5|      true|          300000|
|     TX4|  Produk-19|Rumah Tangga|     0|     false|          450000|
|     TX5|  Produk-10|  Elektronik|     5|      true|          225000|
|     TX6|  Produk-26|   Kesehatan|     4|      true|          450000|
|     TX7|   Produk-4|     Makanan|     0|     false|          150000|
|     TX8|   Produk-7|  Elektronik|     5|      true|          150000|
|     TX9|  Produk-30|  Elektronik|     0|     false|          250000|
+--------+-----------+------------+------+----------+----------------+
only s

D. LOAD (bobot 25%)

Simpan hasil akhir ke HDFS dalam format Parquet, dipartisi berdasarkan kategori, ke path /user/[username]/tugas6/hasil_etl. Verifikasi dengan hdfs dfs -ls -R, lalu baca kembali dan tampilkan count()-nya sebagai bukti data tersimpan utuh.

In [6]:
# Simpan hasil ETL ke HDFS, dipartisi berdasarkan kategori.
path_hdfs_target = "hdfs://localhost:9000/user/caitlyn/tugas6/hasil_etl"

df_enriched.write.mode("overwrite").partitionBy("kategori").parquet(path_hdfs_target)

# Verifikasi struktur folder dan berkas yang tersimpan di HDFS.
!hdfs dfs -ls -R /user/caitlyn/tugas6/hasil_etl

# Baca kembali untuk memastikan data tersimpan utuh.
df_verifikasi = spark.read.parquet(path_hdfs_target)
print("Jumlah baris berhasil disimpan:", df_verifikasi.count())

-rw-r--r--   1 caitlyn supergroup          0 2026-09-28 08:44 /user/caitlyn/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - caitlyn supergroup          0 2026-09-28 08:44 /user/caitlyn/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   1 caitlyn supergroup      16386 2026-09-28 08:44 /user/caitlyn/tugas6/hasil_etl/kategori=Elektronik/part-00000-e29d1fb0-bfc6-4856-9045-15e058a160d4.c000.snappy.parquet
drwxr-xr-x   - caitlyn supergroup          0 2026-09-28 08:44 /user/caitlyn/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   1 caitlyn supergroup      11158 2026-09-28 08:44 /user/caitlyn/tugas6/hasil_etl/kategori=Fashion/part-00000-e29d1fb0-bfc6-4856-9045-15e058a160d4.c000.snappy.parquet
drwxr-xr-x   - caitlyn supergroup          0 2026-09-28 08:44 /user/caitlyn/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   1 caitlyn supergroup      10418 2026-09-28 08:44 /user/caitlyn/tugas6/hasil_etl/kategori=Kesehatan/part-00000-e29d1fb0-bfc6-4856-9045-15e058a160d4.c000.snappy.parquet
drwxr-xr-x   - caitlyn s

E. Insight Akhir (bobot 15%)

Dari data hasil ETL, tampilkan (menggunakan DataFrame API atau Spark SQL, bebas memilih): kategori produk mana yang memiliki persentase transaksi dengan ulasan (ada_ulasan = True) paling rendah? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

In [7]:
# Hitung persentase transaksi dengan ulasan per kategori.
df_insight = df_verifikasi.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    count(when(col("ada_ulasan") == True, True)).alias("transaksi_dengan_ulasan")
).withColumn(
    "persentase_ulasan",
    round((col("transaksi_dengan_ulasan") / col("total_transaksi")) * 100, 2)
).orderBy(col("persentase_ulasan").asc())

df_insight.show()

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|     Makanan|            536|                    369|            68.84|
|   Kesehatan|            961|                    662|            68.89|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|  Elektronik|           1653|                   1168|            70.66|
+------------+---------------+-----------------------+-----------------+



Interpretasi Bisnis:
Kategori dengan persentase ulasan paling rendah merupakan titik buta (blind spot) bagi tim marketing karena minimnya social proof dan feedback produk. Hal ini penting diketahui agar tim marketing dapat meluncurkan kampanye yang ditargetkan (misalnya voucher atau insentif) khusus bagi konsumen di kategori tersebut agar mereka bersedia meninggalkan ulasan pada pembelian berikutnya.